# 05 — Segmentation Test Evaluation

Runs batched inference on the held-out **test sets** (20 %) of all 5 folds and produces:

- **Per-fold per-image metric tables** — `outputs/tables/<task>/<dataset>/<exp>/fold_k/fold_k_test_per_image.csv`  
  Columns: `image_id, patient_id, tumor_class, dataset, image_path, mask_path, pred_path,`  
  `dice, iou, sensitivity, precision, pred_mask_area_ratio, gt_mask_area_ratio, area_ratio_delta,`  
  `true_positive_pixels, false_positive_pixels, false_negative_pixels, total_pixels`
- **Per-fold summary tables** — `fold_k_overall.csv`, `fold_k_by_class.csv`
- **Cross-fold aggregation** — `cv_results.csv`, `cv_summary.csv`, `cv_summary_enriched.csv`,
  `cv_by_class.csv`, `cv_class_summary.csv`, `cv_per_image.csv`
- **Per-patient aggregation** — `cv_per_patient.csv`
- **Predicted mask PNGs** — `outputs/predictions/segmentation/<dataset>/<exp>/fold_k/<image_id>.png`
- **Fold manifests** — `manifest.json` per fold + `prediction_manifest.json` experiment-level (§8)

**Prerequisites:** run `03_train.ipynb` for all folds and sync to Drive.

**Runtime:** GPU (T4 or better). Inference is fast — all 5 folds typically finish in < 10 min.

## Cell 1 — Install dependencies

In [1]:
%pip install -q pytorch-lightning segmentation-models-pytorch albumentations opencv-python-headless timm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 42.7 MB/s eta 0:00:00


## Cell 2 — Bootstrap: Drive + repo

In [ ]:
import os, sys

if not os.path.exists("/content/senior_project"):
    from google.colab import userdata
    try:
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = None
    url = "https://github.com/salemaker47/senior_project.git"
    if token:
        url = url.replace("https://", f"https://{token}@", 1)
    os.system(f"git clone {url} /content/senior_project")
if "/content/senior_project" not in sys.path:
    sys.path.insert(0, "/content/senior_project")

from src.notebook_setup import setup_environment

DRIVE_ROOT, REPO_ROOT = setup_environment(
    repo_url="https://github.com/salemaker47/senior_project.git",
)
print(f"DRIVE_ROOT: {DRIVE_ROOT}")
print(f"REPO_ROOT:  {REPO_ROOT}")

In [ ]:
import os, psutil
print(f"CPU count:   {os.cpu_count()}")
print(f"RAM total:   {psutil.virtual_memory().total / 1e9:.1f} GB")
print(f"RAM avail:   {psutil.virtual_memory().available / 1e9:.1f} GB")

## Cell 3 — EXPERIMENT to evaluate

Set `RECIPE`, `DATASET`, and `SPLIT_SCHEME` to match the experiment you trained in `03_train.ipynb`.

- `RECIPE` must be one of the 7 reference recipes (see `configs/seg/reference_experiments.py`).
- `SPLIT_SCHEME` **must** match the scheme used during training — mismatches evaluate on the wrong test set.
- `FOLD_TO_RUN` / `RUN_ALL_FOLDS`: run one fold first to spot-check, then set `RUN_ALL_FOLDS = True`.

In [ ]:
import torch
from configs.seg.reference_experiments import get_experiment, REFERENCE_RECIPES

# ── What to evaluate ──────────────────────────────────────────────────────
RECIPE       = "03_dicebce"      # one of REFERENCE_RECIPES
DATASET      = "brats2020"        # "figshare" | "brats2020"
SPLIT_SCHEME = "image_level"     # "image_level" | "patient_level"

EXPERIMENT = get_experiment(RECIPE, dataset=DATASET, split_scheme=SPLIT_SCHEME, fold=1)

assert EXPERIMENT["task"] == "segmentation"
assert EXPERIMENT["dataset"] in ("figshare", "brats2020")
assert EXPERIMENT["split_scheme"] in ("image_level", "patient_level")

# ── Evaluation settings ───────────────────────────────────────────────────
SAVE_PNG_PREDS = True    # save predicted mask PNGs to outputs/predictions/...
BATCH_SIZE     = 32
NUM_WORKERS    = 2
REFERENCE_DICE = None    # None = auto-lookup from sg_eval_utils.REFERENCE_DICE

# ── Fold control ──────────────────────────────────────────────────────────
FOLD_TO_RUN   = 1       # used when RUN_ALL_FOLDS = False
RUN_ALL_FOLDS = True    # flip to True after fold 1 looks good

device       = "cuda" if torch.cuda.is_available() else "cpu"
folds_to_run = list(range(1, 6)) if RUN_ALL_FOLDS else [FOLD_TO_RUN]

print(f"experiment   : {EXPERIMENT['name']}")
print(f"dataset      : {EXPERIMENT['dataset']}")
print(f"split_scheme : {EXPERIMENT['split_scheme']}")
print(f"model        : {EXPERIMENT['model_name']}")
print(f"device       : {device}")
print(f"folds        : {folds_to_run}")
print(f"\nAvailable recipes: {REFERENCE_RECIPES}")

## Cell 4 — Copy dataset + checkpoints to local SSD

`copy_to_local` auto-detects a zip at `DRIVE_ROOT/data/<dataset>.zip` and extracts locally (~2–3 min).  
Falls back to `shutil.copytree` when no zip exists.

Checkpoints are also copied to local SSD so model loads (~80 MB × 5 folds) don't stream through Drive FUSE.
All predictions and tables are written to local SSD and synced to Drive in Cell 12.

In [ ]:
import shutil, time
from pathlib import Path

from src.train_utils    import set_global_seed
from src.notebook_setup import copy_to_local

set_global_seed(EXPERIMENT["seed"])
LOCAL_ROOT = copy_to_local(DRIVE_ROOT, datasets=[EXPERIMENT["dataset"]])
print(f"LOCAL_ROOT: {LOCAL_ROOT}")

# ── Copy checkpoints from Drive to local SSD ──────────────────────────────
ckpt_src = (
    DRIVE_ROOT / "outputs" / "checkpoints"
    / EXPERIMENT["task"] / EXPERIMENT["dataset"] / EXPERIMENT["name"]
)
ckpt_dst = (
    LOCAL_ROOT / "outputs" / "checkpoints"
    / EXPERIMENT["task"] / EXPERIMENT["dataset"] / EXPERIMENT["name"]
)

if ckpt_dst.exists():
    n_pt = len(list(ckpt_dst.rglob("best_model.pt")))
    print(f"[checkpoints] already local ({n_pt} folds): {ckpt_dst}")
elif ckpt_src.exists():
    t0 = time.time()
    ckpt_dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(str(ckpt_src), str(ckpt_dst))
    n_pt = len(list(ckpt_dst.rglob("best_model.pt")))
    print(f"[checkpoints] copied in {time.time()-t0:.1f}s ({n_pt} folds)")
else:
    print(f"[checkpoints] WARNING: not found at {ckpt_src}")
    print(f"  Train this experiment first (03_train.ipynb), then sync to Drive.")

## Cell 5 — Sanity check: split_scheme vs training config

Reads `experiment_config.json` from fold 1 and verifies `split_scheme` matches. A mismatch means
you would be evaluating on the wrong test set.

In [ ]:
import json

cfg_path = (
    LOCAL_ROOT / "outputs" / "checkpoints"
    / EXPERIMENT["task"] / EXPERIMENT["dataset"] / EXPERIMENT["name"]
    / "fold_1" / "experiment_config.json"
)

if cfg_path.exists():
    cfg = json.loads(cfg_path.read_text())
    # experiment_config.json stores the config under either "EXPERIMENT" or "experiment" key
    saved_exp = cfg.get("EXPERIMENT") or cfg.get("experiment") or cfg
    saved_scheme = saved_exp.get("split_scheme")
    saved_model  = saved_exp.get("model_name")
    if saved_scheme and saved_scheme != EXPERIMENT["split_scheme"]:
        print(
            f"  WARNING: experiment_config says split_scheme={saved_scheme!r} "
            f"but SPLIT_SCHEME={EXPERIMENT['split_scheme']!r}\n"
            f"  Fix SPLIT_SCHEME in Cell 3 before running evaluation."
        )
    else:
        print(f"  split_scheme : OK ({saved_scheme!r})")
    if saved_model and saved_model != EXPERIMENT["model_name"]:
        print(f"  NOTE: checkpoint model_name={saved_model!r}, EXPERIMENT has {EXPERIMENT['model_name']!r}")
    else:
        print(f"  model_name   : OK ({saved_model!r})")
else:
    print(f"  experiment_config.json not found: {cfg_path}")
    print(f"  (checkpoints may not have been copied — check Cell 4 output above)")

## Cell 6 — `evaluate_one_fold` function

Each call:
1. Loads `best_model.pt` from local SSD.
2. Runs batched inference over the fold's test CSV.
3. Saves predicted mask PNGs to `outputs/predictions/segmentation/<dataset>/<exp>/fold_k/`.
4. Computes the full per-image metric suite (dice, iou, sensitivity, precision, area ratios, pixel counts).
5. Writes the per-fold `manifest.json` (§8 integrity contract).
6. Returns per-fold summary tables consumed by the aggregation cells below.

In [ ]:
import time

from src.data_utils    import load_metadata, validate_metadata
from src.sg_test_utils import load_model_from_pt, evaluate_fold
from src.sg_eval_utils import summarize_fold_results
from src.file_utils    import (
    experiment_paths,
    fold_split_csv_paths,
    seg_predictions_dir,
)


def evaluate_one_fold(fold: int):
    print(f"\n{'='*60}\n  FOLD {fold}\n{'='*60}")
    t0 = time.time()

    # Per-fold paths on LOCAL_ROOT (checkpoints were copied in Cell 4)
    fold_paths = experiment_paths(
        LOCAL_ROOT,
        task=EXPERIMENT["task"],
        dataset=EXPERIMENT["dataset"],
        experiment_name=EXPERIMENT["name"],
        fold=fold,
    )
    pt_path = fold_paths["best_model"]
    if not pt_path.exists():
        print(f"  best_model.pt missing — skipping: {pt_path}")
        print(f"  (check Cell 4: did checkpoints copy successfully?)")
        return None

    # Test CSV paths (data was copied to LOCAL_ROOT in Cell 4)
    csv_paths = fold_split_csv_paths(
        LOCAL_ROOT,
        dataset=EXPERIMENT["dataset"],
        scheme=EXPERIMENT["split_scheme"],
        fold=fold,
    )
    test_df = load_metadata(csv_paths["test"]).reset_index(drop=True)
    validate_metadata(test_df)
    print(f"  test rows: {len(test_df)}  split_scheme={EXPERIMENT['split_scheme']}")

    # Predictions written to LOCAL_ROOT (synced to Drive in Cell 12)
    pred_dir = seg_predictions_dir(
        LOCAL_ROOT,
        dataset=EXPERIMENT["dataset"],
        seg_experiment_name=EXPERIMENT["name"],
        fold=fold,
    )

    model, _ = load_model_from_pt(pt_path=pt_path, device=device)

    result = evaluate_fold(
        model=model,
        test_df=test_df,
        project_root=LOCAL_ROOT,
        predictions_dir=pred_dir,
        fold=fold,
        experiment_name=EXPERIMENT["name"],
        dataset=EXPERIMENT["dataset"],
        split_scheme=EXPERIMENT["split_scheme"],
        checkpoint_path=pt_path,
        test_csv_path=csv_paths["test"],
        model_name=EXPERIMENT["model_name"],
        encoder_weights=EXPERIMENT.get("encoder_weights"),
        image_size=EXPERIMENT["image_size"],
        preprocessing=EXPERIMENT["preprocessing"],
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        device=device,
        threshold=EXPERIMENT["threshold"],
        save_pngs=SAVE_PNG_PREDS,
    )

    per_image_df = result["per_image_df"]
    micro        = result["micro_counts"]

    # Save per-fold per-image CSV
    per_img_csv = fold_paths["tables"] / f"fold_{fold}_test_per_image.csv"
    per_image_df.to_csv(per_img_csv, index=False)
    print(f"  saved per-image table -> {per_img_csv}")

    summaries = summarize_fold_results(
        per_image_df=per_image_df,
        fold=fold,
        experiment_name=EXPERIMENT["name"],
        micro_counts=micro,
    )
    summaries["fold_overall" ].to_csv(fold_paths["tables"] / f"fold_{fold}_overall.csv",   index=False)
    summaries["fold_by_class"].to_csv(fold_paths["tables"] / f"fold_{fold}_by_class.csv",  index=False)

    ov = summaries["fold_overall"].iloc[0]
    print(f"  elapsed     : {time.time()-t0:.1f}s")
    print(f"  dice macro  : {ov['dice_macro']:.4f} \u00b1 {ov['dice_macro_std']:.4f}")
    print(f"  dice micro  : {ov['dice_micro']:.4f}")
    print(f"  iou  macro  : {ov['iou_macro']:.4f}")
    print(f"  iou  micro  : {ov['iou_micro']:.4f}")
    print(f"  tp/fp/fn/tn : {ov['total_tp']}/{ov['total_fp']}/{ov['total_fn']}/{ov['total_tn']}")
    return summaries


print("evaluate_one_fold defined.")

## Cell 7 — Per-fold evaluation loop

Runs `evaluate_one_fold` for each fold in `folds_to_run`. Failed folds are skipped with a
printed traceback; the loop always completes.

In [ ]:
import traceback

fold_summaries = []
for f in folds_to_run:
    try:
        s = evaluate_one_fold(f)
        if s is not None:
            fold_summaries.append(s)
    except Exception as e:
        print(f"  \u2717 fold {f} FAILED: {type(e).__name__}: {e}")
        traceback.print_exc()

print(f"\nevaluated {len(fold_summaries)} / {len(folds_to_run)} folds successfully")

## Cell 8 — Cross-fold aggregation

Combines per-fold summaries into the six cross-fold CSVs expected under
`outputs/tables/<task>/<dataset>/<exp>/`.  
Also displays a styled per-fold table inline.

In [ ]:
import pandas as pd
from IPython.display import display

from src.sg_eval_utils import aggregate_cv_results
from src.file_utils    import experiment_root_paths, save_json

if not fold_summaries:
    raise RuntimeError("No fold summaries — all folds failed. Check Cell 7 output.")

agg = aggregate_cv_results(
    fold_summaries=fold_summaries,
    experiment_name=EXPERIMENT["name"],
    dataset=EXPERIMENT["dataset"],
    split_scheme=EXPERIMENT["split_scheme"],
    reference_dice=REFERENCE_DICE,
)

cv_results          = agg["cv_results"]
cv_summary          = agg["cv_summary"]
cv_summary_enriched = agg["cv_summary_enriched"]
cv_by_class         = agg["cv_by_class"]
cv_class_summary    = agg["cv_class_summary"]
cv_per_image        = agg["cv_per_image"]

root_paths = experiment_root_paths(
    LOCAL_ROOT,
    task=EXPERIMENT["task"],
    dataset=EXPERIMENT["dataset"],
    experiment_name=EXPERIMENT["name"],
)
out_dir = root_paths["tables"]

cv_results.to_csv          (out_dir / "cv_results.csv",          index=False)
cv_summary.to_csv          (out_dir / "cv_summary.csv",          index=False)
cv_summary_enriched.to_csv (out_dir / "cv_summary_enriched.csv", index=False)
cv_by_class.to_csv         (out_dir / "cv_by_class.csv",         index=False)
cv_class_summary.to_csv    (out_dir / "cv_class_summary.csv",    index=False)
cv_per_image.to_csv        (out_dir / "cv_per_image.csv",        index=False)
print(f"saved 6 CV tables -> {out_dir}")

# ── Styled per-fold table ──────────────────────────────────────────────────
_show_cols = [c for c in
    ["fold", "n_images", "dice_macro", "dice_micro", "iou_macro", "iou_micro",
     "sensitivity_macro", "precision_macro"]
    if c in cv_results.columns]

_float_cols = [c for c in _show_cols if c not in ("fold", "n_images")]

def _dice_color(col):
    return col.map(
        lambda v: "background-color: #c8e6c9" if v >= 0.80
        else "background-color: #fff9c4" if v >= 0.60
        else "background-color: #ffcdd2"
    )

styled = (
    cv_results[_show_cols].style
    .format({c: "{:.4f}" for c in _float_cols})
    .apply(_dice_color, subset=[c for c in ["dice_macro", "dice_micro"] if c in _show_cols])
    .set_caption(
        f"{EXPERIMENT['name']} \u00b7 {EXPERIMENT['dataset']} \u00b7 {EXPERIMENT['split_scheme']}"
        " \u2014 per-fold test results"
    )
)
display(styled)

## Cell 9 — Per-patient aggregation

Aggregates per-image metrics to the patient level across all folds. Useful for spotting
patients with consistently poor or good segmentation.

In [ ]:
from src.sg_eval_utils import aggregate_cv_per_patient

cv_per_patient = aggregate_cv_per_patient(cv_per_image)
cv_per_patient.to_csv(out_dir / "cv_per_patient.csv", index=False)
print(f"  saved cv_per_patient.csv: {len(cv_per_patient)} patients")

if not cv_per_patient.empty:
    _pfmt = {c: "{:.4f}" for c in
             ["dice_mean","dice_median","dice_min","dice_max",
              "iou_mean","iou_median","sensitivity_mean","precision_mean"]}
    print("\nTop-10 patients by mean Dice:")
    display(cv_per_patient.nlargest(10, "dice_mean").style.format(_pfmt, na_rep="\u2014"))
    print("\nBottom-10 patients by mean Dice:")
    display(cv_per_patient.nsmallest(10, "dice_mean").style.format(_pfmt, na_rep="\u2014"))

## Cell 10 — Experiment manifest (§8)

Walks the per-fold `manifest.json` files written by `evaluate_fold` and produces the
experiment-level `prediction_manifest.json` at `outputs/predictions/segmentation/<dataset>/<exp>/`.

In [ ]:
from src.sg_test_utils import write_experiment_manifest

manifest_path, manifest = write_experiment_manifest(
    project_root=LOCAL_ROOT,
    dataset=EXPERIMENT["dataset"],
    experiment_name=EXPERIMENT["name"],
    expected_folds=tuple(folds_to_run),
)
print(f"  manifest path      : {manifest_path}")
print(f"  folds present      : {manifest['folds_present']}")
print(f"  total predictions  : {manifest['total_predictions']}")
print(f"  all covered        : {manifest['all_test_images_covered']}")

## Cell 11 — Report-style readout + evaluation config

Prints the headline numbers in the format used in the project report. Also saves
`test_eval_config.json` for traceability.

In [ ]:
row = cv_summary.iloc[0]

print("=" * 60)
print(f"  {EXPERIMENT['name']}")
print(f"  dataset={EXPERIMENT['dataset']}  split={EXPERIMENT['split_scheme']}")
print("=" * 60)
print(f"\nDice (micro) : {row['report_dice_micro']}")
print(f"Dice (macro) : {row['report_dice_macro']}")
print(f"IoU  (micro) : {row['report_iou_micro']}")
print(f"IoU  (macro) : {row['report_iou_macro']}")

delta = row.get("delta_vs_reference")
if delta is not None:
    sign = "+" if delta >= 0 else ""
    print(f"vs reference : {sign}{delta:.4f}  (ref={row['reference_dice']:.4f})")

print("\nPer-fold:")
_cols = [c for c in ["fold","n_images","dice_macro","dice_micro","iou_macro","iou_micro"]
         if c in cv_results.columns]
print(cv_results[_cols].to_string(index=False))

if not cv_class_summary.empty:
    print("\nPer-class (micro Dice across folds):")
    _ccols = [c for c in
        ["tumor_class","n_test_images_total","dice_micro_mean","dice_micro_std","report_dice_micro"]
        if c in cv_class_summary.columns]
    print(cv_class_summary[_ccols].to_string(index=False))

# Styled class summary
if not cv_class_summary.empty:
    _cfmt = {c: "{:.4f}" for c in ["dice_micro_mean","dice_micro_std",
                                     "dice_macro_mean","iou_micro_mean","iou_macro_mean"]
             if c in cv_class_summary.columns}
    display(cv_class_summary.style.format(_cfmt, na_rep="\u2014")
            .set_caption("Per-class summary (cross-fold)"))

# Save evaluation config for traceability
save_json(
    {
        "experiment_name":     EXPERIMENT["name"],
        "recipe":              EXPERIMENT.get("recipe"),
        "dataset":             EXPERIMENT["dataset"],
        "split_scheme":        EXPERIMENT["split_scheme"],
        "model_name":          EXPERIMENT["model_name"],
        "preprocessing":       EXPERIMENT["preprocessing"],
        "image_size":          EXPERIMENT["image_size"],
        "threshold":           EXPERIMENT["threshold"],
        "batch_size":          BATCH_SIZE,
        "save_pngs":           SAVE_PNG_PREDS,
        "folds_evaluated":     [int(f) for f in cv_results["fold"].tolist()],
        "n_test_images_total": int(row["n_test_images_total"]),
        "dice_micro_mean":     float(row["dice_micro_mean"]),
        "dice_micro_std":      float(row["dice_micro_std"]),
        "dice_macro_mean":     float(row["dice_macro_mean"]),
        "dice_macro_std":      float(row["dice_macro_std"]),
        "delta_vs_reference":  float(delta) if delta is not None else None,
    },
    out_dir / "test_eval_config.json",
)
print(f"\nsaved test_eval_config.json -> {out_dir}")

## Cell 12 — Sync to Drive

Copies `predictions/` and `tables/` for this experiment from local SSD back to Drive.  
Run this cell once all folds have evaluated successfully.

In [ ]:
from src.notebook_setup import sync_outputs_to_drive

sync_outputs_to_drive(
    drive_root=DRIVE_ROOT,
    local_root=LOCAL_ROOT,
    task=EXPERIMENT["task"],
    dataset=EXPERIMENT["dataset"],
    experiment_name=EXPERIMENT["name"],
    categories=["predictions", "tables"],
)
print(f"Drive output root: {DRIVE_ROOT / 'outputs'}")